# Training first model

In [ ]:
# UTAT Space Systems Deep Learning On-Boarding Project, Task 2
# Ethan Vu

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import joblib
import csv
from pathlib import Path

data_path = Path('../datasets/simpler_data_rwc.csv')
val_data_path = Path('../validation-data/m1-val-r2.csv')
test_data_path = Path('../testing-data/m1-test-r2.csv')

data = pd.read_csv(data_path)
# data = pd.read_csv('simpler_data_rwc.csv')
wavelength_cols = data.columns[data.columns.str.isdigit()]

X = data[wavelength_cols].values
y = data[["gv_fraction", "npv_fraction", "soil_fraction"]]

# 70/15/15 train/validation/test split
X_train, X_val_test, y_train, y_val_test = train_test_split(X, y, test_size=0.3)
X_val, X_test, y_val, y_test = train_test_split(X_val_test, y_val_test, test_size=0.5)

scalar = StandardScaler()

X_train_s = scalar.fit_transform(X_train)
X_test_s = scalar.transform(X_test)
X_val_s = scalar.transform(X_val)
y_train_s = scalar.fit_transform(y_train)
y_test_s = scalar.transform(y_test)
y_val_s = scalar.transform(y_val)

X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)
X_val_t = torch.tensor(X_val_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train.to_numpy(dtype=np.float32), dtype=torch.float32)
y_test_t = torch.tensor(y_test.to_numpy(dtype=np.float32), dtype=torch.float32)
y_val_t = torch.tensor(y_val.to_numpy(dtype=np.float32), dtype=torch.float32)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

class Unmixer (nn.Module):
    def __init__(self):
        super(Unmixer, self).__init__()

        self.fc1 = nn.Linear(210, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 3)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.softmax(self.fc3(x), dim=1)

        return x

model = Unmixer()
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)

epochs = 100
train_losses = []
val_losses = []
val_r2_scores = []

for epoch in range(epochs):
    model.train()
    avg_train_loss = 0
    running_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)

    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # validate and record losses and r^2
    with torch.no_grad():
        model.eval()
        val_preds = model(X_val_t)
        val_loss = criterion(val_preds, y_val_t)

        r2 = r2_score(y_val_t.cpu().numpy(), val_preds.cpu().numpy(), multioutput='raw_values')

        val_r2_scores.append(r2)
        val_losses.append(val_loss.item())

with open(val_data_path, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(val_r2_scores)

# export the model in case we want to preserve it
joblib.dump(model, '../models/unmixer1-1.pkl')

# graph training and validation losses
plt.plot(range(1, epochs+1), train_losses, label="Train Loss")
plt.plot(range(1, epochs+1), val_losses, label="Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Mean Squared Error Loss")
# plt.title("Training and Validation Losses of First Model")

plt.legend()


# Testing first model

In [ ]:
# load the exported model for funsies

try:
    unmixer = joblib.load('../models/unmixer-1.pkl')
except FileNotFoundError:
    unmixer = model

with torch.no_grad():
    unmixer.eval()

    preds = unmixer(X_test_t)
    loss = criterion(preds, y_test_t)
    y_test_np = y_test_t.cpu().numpy()
    preds_np = preds.cpu().numpy()

    test_r2 = r2_score(y_test_np, preds_np, multioutput='raw_values')
    avg_test_r2 = np.mean(test_r2)

print(f"""GV R^2: {test_r2[0]}
NPV R^2: {test_r2[1]}
Soil R^2: {test_r2[2]}
Overall R^2: {avg_test_r2}""")

with open(test_data_path, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['GV', 'NPV', 'Soil', 'Overall'])
    writer.writerow([test_r2[0], test_r2[1], test_r2[2], avg_test_r2])

# graph testing R^2

plt.scatter(preds_np[:,0], y_test_np[:,0], color='green', label="GV")
plt.scatter(preds_np[:,1], y_test_np[:,1], color='orange', label="NPV")
plt.scatter(preds_np[:,2], y_test_np[:,2],color='blue', label="Soil")

overall_y_test = []
overall_y_test.extend(y_test_np[:,0])
overall_y_test.extend(y_test_np[:,1])
overall_y_test.extend(y_test_np[:,2])
overall_preds = []
overall_preds.extend(preds_np[:,0])
overall_preds.extend(preds_np[:,1])
overall_preds.extend(preds_np[:,2])

coefficients = np.polyfit(overall_y_test, overall_preds, 1)
polynomial = np.poly1d(coefficients)
trendline = polynomial(overall_y_test)
plt.plot([0,1], [0,1], linestyle='--', color='white', label='Perfect Prediction')
plt.plot(trendline, overall_y_test, color='red', linestyle='--', label='Trendline')

print(f'Trendline slope: {coefficients[0]}, y-intercept: {coefficients[1]}, x-intercept: {-coefficients[0]+trendline[1]}')

plt.xlabel("Predicted abundance")
plt.ylabel("True abundance")
# plt.title("True vs Predicted Abundances of First Model")
plt.legend()

# Plot Validation R^2

In [ ]:
gv_val_r2 = []
npv_val_r2 = []
soil_val_r2 = []

for score in val_r2_scores:
    gv_val_r2.append(score[0])
    npv_val_r2.append(score[1])
    soil_val_r2.append(score[2])


plt.plot(range(1, 101), gv_val_r2, label="GV", color='green')
plt.plot(range(1, 101), npv_val_r2, label="NPV", color='orange')
plt.plot(range(1, 101), soil_val_r2, label="Soil", color='blue')
plt.xlabel("Epochs")
plt.ylabel("Validation R^2")
plt.legend()